# RegressLM — missing-evidence repair (Colab GPU)

This notebook produces the two pieces the official judge found missing:

1. **Claim 1 / ONNX accuracy:** the same released unified checkpoint is evaluated on released `GraphArch-Regression` ONNX strings with `val_accuracy` targets.
2. **Claim 3 / execution provenance:** all 17 scored CodeNet languages are evaluated at 200 rows/language with visible model-loading, fetch, batch-progress, timing, seed, and environment logs.

Use a **T4, L4, or A100 GPU runtime**, then choose **Runtime → Run all**. The final cell downloads `regresslm_evidence_bundle.zip`; send that ZIP back to Codex. APPS and KBSS are disabled by default because the official judge already accepted their n=512 results, but one switch below enables them.


In [ ]:
# Configuration: the defaults directly target the two rejected claims.
RUN_CODENET_17 = True
RUN_ONNX_ACCURACY = True
RUN_ALREADY_ACCEPTED_APPS_KBSS = False

ROWS_PER_LANGUAGE = 200
ONNX_ROWS_PER_SPACE = 64
# The current HF indexed conversion is partial but contains NASBench101. This
# supplies the missing real accuracy metric without a 4.1 GB parquet download.
ONNX_SPACES = ["NASBench101"]
NUM_SAMPLES = 8
BATCH_SIZE = 16
SEED = 42
print({k: v for k, v in globals().items() if k.startswith("RUN_") or k in {
    "ROWS_PER_LANGUAGE", "ONNX_ROWS_PER_SPACE", "ONNX_SPACES", "NUM_SAMPLES", "BATCH_SIZE", "SEED"
}})


In [ ]:
import os, sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "regress-lm[extras]", "pyarrow", "scipy", "pandas", "requests"], check=True)
# The checkpoint declares this exact version. Transformers 5.x makes this custom
# encoder-decoder input-insensitive, so install 4.53.2 last.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.53.2"], check=True)

import csv, glob, hashlib, json, math, platform, random, shutil, time, urllib.request, zipfile
from ast import literal_eval
from pathlib import Path
import numpy as np, pandas as pd, pyarrow as pa, pyarrow.dataset as ds, requests, torch
from scipy import stats
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

assert torch.cuda.is_available(), "Select a GPU runtime: Runtime > Change runtime type > T4/L4/A100"
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = "cuda"
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
RESULTS = Path("regresslm_evidence"); RESULTS.mkdir(exist_ok=True)
environment = {
    "python": sys.version, "platform": platform.platform(), "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda, "gpu": torch.cuda.get_device_name(0),
    "transformers": __import__("transformers").__version__, "dtype": str(dtype), "seed": SEED,
}
print(json.dumps(environment, indent=2), flush=True)


In [ ]:
# Load the public checkpoint locally so its bundled encoder tokenizer is used.
from huggingface_hub import snapshot_download, hf_hub_download
REPO = "akhauriyash/RegressLM-gemma-s-RLM-table3"
CKPT_DIR = snapshot_download(REPO)
PATCH_RAW = "https://raw.githubusercontent.com/MachineLearning-Nerd/icml26-repro-utTapVWtc7/master/repro/patches"
for fn in ["configuration_regresslm.py", "modeling_regresslm.py"]:
    urllib.request.urlretrieve(f"{PATCH_RAW}/{fn}", os.path.join(CKPT_DIR, fn))
for d in glob.glob(os.path.expanduser("~/.cache/huggingface/modules/transformers_modules/*")):
    if os.path.exists(os.path.join(d, "configuration_regresslm.py")):
        shutil.rmtree(d, ignore_errors=True)

t0 = time.time()
tok = AutoTokenizer.from_pretrained(CKPT_DIR, trust_remote_code=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    CKPT_DIR, trust_remote_code=True, torch_dtype=dtype
).to(device).eval()
N_OUT = int(model.config.num_tokens_per_obj) * int(model.config.max_num_objs)
model_info = {
    "repo": REPO, "checkpoint_commit": Path(CKPT_DIR).name,
    "parameters": sum(p.numel() for p in model.parameters()), "n_out_tokens": N_OUT,
    "load_seconds": time.time() - t0,
    "config_sha256": hashlib.sha256(Path(CKPT_DIR, "config.json").read_bytes()).hexdigest(),
}
print(json.dumps(model_info, indent=2), flush=True)


In [ ]:
def decode(ids):
    value = tok.token_ids_to_floats(ids)
    return float(value[0] if isinstance(value, (list, tuple)) else value)

@torch.inference_mode()
def evaluate(task, inputs, targets, max_length=2048, batch_size=BATCH_SIZE):
    started = time.time(); records = []
    ordered = sorted(enumerate(zip(inputs, targets)), key=lambda pair: len(pair[1][0]))
    for start in range(0, len(ordered), batch_size):
        indexed_chunk = ordered[start:start+batch_size]
        original_indices = [index for index, _ in indexed_chunk]
        chunk = [pair[0] for _, pair in indexed_chunk]
        chunk_targets = [pair[1] for _, pair in indexed_chunk]
        enc = tok(chunk, return_tensors="pt", truncation=True, padding=True,
                  max_length=max_length).to(device)
        # Reuse one encoder pass for all 8 stochastic decodes.
        out = model.generate(
            **enc, do_sample=True, top_p=0.95, temperature=1.0,
            min_new_tokens=N_OUT, max_new_tokens=N_OUT,
            num_return_sequences=NUM_SAMPLES,
            pad_token_id=getattr(tok, "pad_token_id", 0), use_cache=True,
        )
        seqs = out.reshape(len(chunk), NUM_SAMPLES, -1)
        draws = np.asarray([
            [decode(seqs[row, sample].tolist()) for sample in range(NUM_SAMPLES)]
            for row in range(len(chunk))
        ], dtype=float)
        preds = np.nanmedian(draws, axis=1)
        for j, (index, text, target, pred) in enumerate(zip(
                original_indices, chunk, chunk_targets, preds)):
            records.append({
                "task": task, "i": index,
                "input_sha256": hashlib.sha256(text.encode()).hexdigest(),
                "target": float(target), "prediction": float(pred),
                **{f"draw_{d}": float(draws[j, d]) for d in range(NUM_SAMPLES)},
            })
        print(f"[{task}] batch {start//batch_size+1}/{math.ceil(len(inputs)/batch_size)} "
              f"({start+len(chunk)}/{len(inputs)}) elapsed={time.time()-started:.1f}s", flush=True)
    frame = pd.DataFrame(records).sort_values("i").reset_index(drop=True)
    valid = np.isfinite(frame.target) & np.isfinite(frame.prediction)
    rho = float(stats.spearmanr(frame.loc[valid, "target"], frame.loc[valid, "prediction"]).statistic)
    frame.to_csv(RESULTS / f"{task}.csv", index=False)
    result = {"task": task, "n": int(valid.sum()), "spearman": rho,
              "seconds": time.time()-started, "num_samples": NUM_SAMPLES}
    print("RESULT", json.dumps(result), flush=True)
    return result


In [ ]:
# Claim 3: server-side SQL returns only the 3,400 evaluated rows (no 5.6 GB download).
CODENET_17 = ["C++", "Python", "Java", "C", "Ruby", "C#", "Rust", "Go", "Haskell",
              "Kotlin", "JavaScript", "PHP", "D", "Scala", "OCaml", "Perl", "Fortran"]
codenet_results = []
if RUN_CODENET_17:
    url = "https://datasets-server.huggingface.co/filter"
    buckets = {lang: [] for lang in CODENET_17}
    for li, lang in enumerate(CODENET_17):
        escaped = lang.replace("'", "''")
        where = (f'"space"=\'CDSS\' AND "metadata" LIKE '
                 f"'%''language'': ''{escaped}''%'")
        offset = 0
        while len(buckets[lang]) < ROWS_PER_LANGUAGE:
            length = min(100, ROWS_PER_LANGUAGE - len(buckets[lang]))
            response = requests.get(url, params={
                "dataset": "akhauriyash/Code-Regression", "config": "default", "split": "train",
                "where": where, "offset": offset, "length": length,
            }, timeout=600)
            response.raise_for_status(); rows = response.json().get("rows", [])
            assert rows, (lang, offset)
            for item in rows:
                row = item["row"]
                buckets[lang].append((
                    f"# CDSS\n# Language: {lang}\n{row['input']}", float(row["target"])
                ))
            offset += len(rows)
        print(f"server fetch [{li+1}/17] {lang}: {len(buckets[lang])}", flush=True)
    assert all(len(v) == ROWS_PER_LANGUAGE for v in buckets.values())
    for lang in CODENET_17:
        pairs = buckets[lang]
        codenet_results.append(evaluate(
            "codenet_" + lang.replace("+", "p").replace("#", "sharp").lower(),
            [x for x, _ in pairs], [y for _, y in pairs], max_length=2048,
        ) | {"language": lang})
    avg = float(np.mean([r["spearman"] for r in codenet_results]))
    print(f"CLAIM 3: average Spearman across 17 languages = {avg:.6f} (>0.5 required)", flush=True)


In [ ]:
# Claim 1 missing metric: server-side filtering avoids downloading the 4.1 GB ONNX parquet.
def fetch_grapharch(space, limit):
    url = "https://datasets-server.huggingface.co/filter"
    # Spaces around '=' avoid an intermittent dataset-server index-loading bug.
    where = '"space" = ' + repr(space)
    params = {"dataset": "akhauriyash/GraphArch-Regression", "config": "default",
              "split": "train", "where": where, "offset": 0, "length": limit}
    response = requests.get(url, params=params, timeout=600); response.raise_for_status()
    rows = response.json()["rows"]
    if len(rows) != limit:
        raise RuntimeError(
            f"server index has {len(rows)}/{limit} {space} rows; use NASBench101 "
            "or explicitly download the 4.1 GB original parquet")
    return [f"{space}\n\n{r['row']['input']}" for r in rows], [float(r["row"]["val_accuracy"]) for r in rows]

onnx_results = []
if RUN_ONNX_ACCURACY:
    for space in ONNX_SPACES:
        inputs, targets = fetch_grapharch(space, ONNX_ROWS_PER_SPACE)
        print(f"ONNX fetch {space}: n={len(inputs)}, target range={min(targets):.3f}..{max(targets):.3f}", flush=True)
        # ONNX strings are much longer than source-code rows; batch 2 is safe on T4.
        onnx_results.append(evaluate(
            "onnx_" + space.lower(), inputs, targets, max_length=4096, batch_size=2
        ) | {"space": space})
    print("CLAIM 1 ONNX accuracy results", json.dumps(onnx_results, indent=2), flush=True)


In [ ]:
# Optional accepted baselines; disabled by default to save GPU time.
accepted_results = []
if RUN_ALREADY_ACCEPTED_APPS_KBSS:
    if "dataset" not in globals():
        parquet = hf_hub_download("akhauriyash/Code-Regression", "data.parquet", repo_type="dataset")
        dataset = ds.dataset(parquet, format="parquet")
    for space in ["APPS", "KBSS"]:
        flt = pa.compute.equal(pa.compute.field("space"), space); pairs = []
        for batch in dataset.scanner(columns=["input", "target"], filter=flt, batch_size=2048).to_batches():
            for inp, target in zip(batch.column("input"), batch.column("target")):
                if inp.is_valid and target.is_valid: pairs.append((f"{space}\n{inp.as_py()}", float(target.as_py())))
                if len(pairs) == 512: break
            if len(pairs) == 512: break
        accepted_results.append(evaluate(space.lower(), [x for x, _ in pairs], [y for _, y in pairs]))


In [ ]:
# Validate, write a self-contained evidence summary, and download one ZIP.
summary = {
    "environment": environment, "model": model_info,
    "configuration": {"rows_per_language": ROWS_PER_LANGUAGE, "onnx_rows_per_space": ONNX_ROWS_PER_SPACE,
                      "num_samples": NUM_SAMPLES, "batch_size": BATCH_SIZE, "seed": SEED},
    "claim_3_codenet": {
        "per_language": codenet_results,
        "average_spearman": float(np.mean([r["spearman"] for r in codenet_results])) if codenet_results else None,
        "threshold": 0.5,
    },
    "claim_1_onnx_accuracy": onnx_results,
    "accepted_optional": accepted_results,
}
if codenet_results:
    assert len(codenet_results) == 17 and all(r["n"] == ROWS_PER_LANGUAGE for r in codenet_results)
    assert summary["claim_3_codenet"]["average_spearman"] > 0.5
if onnx_results:
    assert len(onnx_results) == len(ONNX_SPACES) and all(r["n"] == ONNX_ROWS_PER_SPACE for r in onnx_results)
(RESULTS / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
zip_path = shutil.make_archive("regresslm_evidence_bundle", "zip", RESULTS)
print(json.dumps(summary, indent=2), flush=True)
print("BUNDLE", zip_path, os.path.getsize(zip_path), "bytes", flush=True)
from google.colab import files
files.download(zip_path)
